# XGBoost — Nested Cross-Validation 5×5 con selección de preprocessing dentro del Inner CV

Esta es una **versión experimental nueva**. No reemplaza la corrida anterior; sirve para compararla contra el baseline.

Flujo:

1. Carga las tres representaciones de texto: `normal`, `stem` y `lemma`.
2. Verifica que las tres versiones tengan exactamente las mismas filas y etiquetas.
3. Mantiene **TF-IDF de palabras (1,2) + 9 características lingüísticas**.
4. Ejecuta **Nested Cross-Validation 5×5**:
   - Outer CV = 5 folds.
   - En cada Outer Fold, Optuna trabaja solo con el `outer_train`.
   - Cada trial de Optuna selecciona conjuntamente `preprocessing` + hiperparámetros de XGBoost.
   - Cada trial se evalúa mediante Inner Stratified 5-Fold usando exactamente los mismos índices para los tres preprocesamientos.
   - El mejor preprocessing + hiperparámetros se reentrena con todo el `outer_train` y se evalúa una sola vez en el `outer_test`.
5. Se reportan F1-Macro de train, inner y outer, además del gap de generalización.
6. Se realiza una optimización final sobre todo el `train` oficial para elegir preprocessing + hiperparámetros definitivos.
7. El `test` oficial permanece completamente reservado hasta el final.

> **Objetivo de esta versión:** comprobar si integrar el preprocessing dentro del Inner CV y usar un espacio de búsqueda más regularizado mejora la generalización respecto a la corrida anterior.


In [1]:
# ============================================================
# 1. IMPORTS Y CONFIGURACIÓN GENERAL
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
DATA_DIR = "../data"

VARIANTES = ["mx", "es", "cu"]

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]

PREPROCESAMIENTOS = {
    "normal": "",
    "stem": "_stem",
    "lemma": "_lemma",
}

N_OUTER = 5
N_INNER = 5
N_TRIALS_NESTED = 50
N_TRIALS_FINAL = 50


## 2. Carga de datos

In [2]:
def cargar_split(variante, prep, split="train"):
    sufijo = PREPROCESAMIENTOS[prep]
    ruta = f"{DATA_DIR}/{split}_clean{sufijo}_{variante}.csv"
    return pd.read_csv(ruta)


def preparar_xy(df):
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).copy()
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    return X, y


def cargar_todos_preps(variante, split="train"):
    """Carga normal/stem/lemma y valida que estén alineados por fila y etiqueta."""
    datasets = {}
    y_referencia = None
    n_referencia = None

    for prep in PREPROCESAMIENTOS:
        df = cargar_split(variante, prep, split=split)
        X, y = preparar_xy(df)

        if y_referencia is None:
            y_referencia = y.reset_index(drop=True)
            n_referencia = len(y)
        else:
            if len(y) != n_referencia:
                raise ValueError(
                    f"{variante}/{split}: {prep} tiene {len(y)} filas, "
                    f"pero la referencia tiene {n_referencia}."
                )
            if not np.array_equal(y.to_numpy(), y_referencia.to_numpy()):
                raise ValueError(
                    f"{variante}/{split}: las etiquetas no están alineadas entre preprocesamientos."
                )

        datasets[prep] = X.reset_index(drop=True)

    return datasets, y_referencia


## 3. Representación fija

La representación se mantiene igual que en la corrida anterior para aislar el efecto de los cambios metodológicos:

- TF-IDF de palabras.
- `ngram_range=(1, 2)`.
- `max_features=20000`.
- 9 características lingüísticas.

En esta versión **no se optimiza TF-IDF todavía**. Primero se evalúa preprocessing + regularización de XGBoost.


In [3]:

def crear_preprocesador():
    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=20000,
        lowercase=False,
        sublinear_tf=True,
        dtype=np.float32,
    )

    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )


## 4. Espacio de búsqueda regularizado para Optuna

El espacio se hace más conservador a partir de la evidencia de la corrida anterior, donde aparecieron gaps Train–Outer altos, especialmente en MX.

Se mantienen 50 trials para que la comparación con el experimento anterior sea más justa.


In [4]:
# ============================================================
# 4. ESPACIO DE BÚSQUEDA PARA OPTUNA
# ============================================================

def sugerir_params_xgb(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 15),
        "subsample": trial.suggest_float("subsample", 0.55, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.90),
        "gamma": trial.suggest_float("gamma", 0.5, 8.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 2.5),
    }


def crear_xgboost(params, random_state=RANDOM_STATE):
    return XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
        **params,
    )


def crear_pipeline(params, random_state=RANDOM_STATE):
    return Pipeline(
        steps=[
            ("features", crear_preprocesador()),
            ("xgb", crear_xgboost(params, random_state=random_state)),
        ]
    )


## 5. Nested Cross-Validation 5×5: preprocessing + hiperparámetros dentro del Inner CV

En cada Outer Fold:

- El `outer_test` queda bloqueado.
- Optuna ve únicamente el `outer_train`.
- Cada trial elige un `preprocessing` (`normal`, `stem` o `lemma`) y una configuración de XGBoost.
- La combinación completa se evalúa mediante **Inner Stratified 5-Fold**.
- Los tres preprocesamientos usan exactamente los mismos índices de folds.
- Al terminar los 50 trials, se obtiene `best_prep + best_params`.
- Se ajusta un modelo nuevo con todo el `outer_train` usando esa combinación.
- Se evalúa una sola vez en el `outer_test`.

Así, el Outer CV evalúa también la **decisión de preprocessing**, no solo los hiperparámetros de XGBoost.


In [5]:
def nested_cv_variante(variante, n_trials=N_TRIALS_NESTED):
    X_por_prep, y = cargar_todos_preps(variante, split="train")

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    resultados_outer = []
    studies_outer = {}
    X_ref = X_por_prep["normal"]

    for outer_fold, (idx_train, idx_test) in enumerate(
        outer_cv.split(X_ref, y), start=1
    ):
        print(f"\n{'='*70}")
        print(f"{variante.upper()} — OUTER FOLD {outer_fold}/{N_OUTER}")
        print(f"{'='*70}")

        y_outer_train = y.iloc[idx_train].reset_index(drop=True)
        y_outer_test = y.iloc[idx_test].reset_index(drop=True)

        X_outer_train = {
            prep: X.iloc[idx_train].reset_index(drop=True)
            for prep, X in X_por_prep.items()
        }
        X_outer_test = {
            prep: X.iloc[idx_test].reset_index(drop=True)
            for prep, X in X_por_prep.items()
        }

        inner_cv = StratifiedKFold(
            n_splits=N_INNER,
            shuffle=True,
            random_state=RANDOM_STATE + outer_fold,
        )
        inner_splits = list(
            inner_cv.split(X_outer_train["normal"], y_outer_train)
        )

        def objective(trial):
            prep = trial.suggest_categorical(
                "preprocessing", list(PREPROCESAMIENTOS.keys())
            )
            params = sugerir_params_xgb(trial)
            pipeline = crear_pipeline(
                params, random_state=RANDOM_STATE + outer_fold
            )

            scores = cross_val_score(
                estimator=pipeline,
                X=X_outer_train[prep],
                y=y_outer_train,
                cv=inner_splits,
                scoring="f1_macro",
                n_jobs=1,
            )

            trial.set_user_attr("inner_f1_std", float(scores.std(ddof=1)))
            return float(scores.mean())

        sampler = optuna.samplers.TPESampler(
            seed=RANDOM_STATE + outer_fold
        )
        study = optuna.create_study(
            direction="maximize", sampler=sampler
        )
        study.optimize(
            objective,
            n_trials=n_trials,
            show_progress_bar=True,
        )
        studies_outer[outer_fold] = study

        best_all = study.best_params.copy()
        best_prep = best_all.pop("preprocessing")
        best_params = best_all

        modelo_outer = crear_pipeline(
            best_params,
            random_state=RANDOM_STATE + outer_fold,
        )
        modelo_outer.fit(
            X_outer_train[best_prep], y_outer_train
        )

        pred_train = modelo_outer.predict(X_outer_train[best_prep])
        pred_outer = modelo_outer.predict(X_outer_test[best_prep])

        f1_train = f1_score(y_outer_train, pred_train, average="macro")
        f1_outer = f1_score(y_outer_test, pred_outer, average="macro")
        gap = f1_train - f1_outer

        resultados_outer.append({
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": best_prep,
            "train_f1_macro": f1_train,
            "inner_best_f1": study.best_value,
            "inner_best_std": study.best_trial.user_attrs.get(
                "inner_f1_std", np.nan
            ),
            "outer_f1_macro": f1_outer,
            "generalization_gap": gap,
            "best_params": best_params,
        })

        print(f"Mejor preprocessing: {best_prep}")
        print(f"F1-Macro train: {f1_train:.4f}")
        print(f"Mejor F1 inner: {study.best_value:.4f}")
        print(f"F1-Macro outer: {f1_outer:.4f}")
        print(f"Gap train-outer: {gap:.4f}")

    df_outer = pd.DataFrame(resultados_outer)
    resumen = {
        "variante": variante,
        "f1_train_mean": df_outer["train_f1_macro"].mean(),
        "f1_nested_mean": df_outer["outer_f1_macro"].mean(),
        "f1_nested_std": df_outer["outer_f1_macro"].std(ddof=1),
        "f1_nested_min": df_outer["outer_f1_macro"].min(),
        "f1_nested_max": df_outer["outer_f1_macro"].max(),
        "gap_mean": df_outer["generalization_gap"].mean(),
    }
    return df_outer, resumen, studies_outer


In [6]:
# ============================================================
# 6. EJECUTAR NESTED CV PARA LAS 3 VARIANTES
# ============================================================

nested_detalle = {}
nested_resumen = []
nested_studies = {}

for variante in VARIANTES:
    detalle, resumen, studies = nested_cv_variante(
        variante, n_trials=N_TRIALS_NESTED
    )
    nested_detalle[variante] = detalle
    nested_resumen.append(resumen)
    nested_studies[variante] = studies


df_nested_resumen = pd.DataFrame(nested_resumen)

print("\nRESULTADOS NESTED CV 5×5")
display(
    df_nested_resumen.style.format({
        "f1_train_mean": "{:.4f}",
        "f1_nested_mean": "{:.4f}",
        "f1_nested_std": "{:.4f}",
        "f1_nested_min": "{:.4f}",
        "f1_nested_max": "{:.4f}",
        "gap_mean": "{:.4f}",
    })
)



MX — OUTER FOLD 1/5


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor preprocessing: stem
F1-Macro train: 0.7623
Mejor F1 inner: 0.6091
F1-Macro outer: 0.6085
Gap train-outer: 0.1539

MX — OUTER FOLD 2/5


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor preprocessing: stem
F1-Macro train: 0.9186
Mejor F1 inner: 0.6229
F1-Macro outer: 0.5910
Gap train-outer: 0.3275

MX — OUTER FOLD 3/5


...

  0%|          | 0/50 [00:00<?, ?it/s]

Mejor preprocessing: stem
F1-Macro train: 0.8422
Mejor F1 inner: 0.6654
F1-Macro outer: 0.6427
Gap train-outer: 0.1996

CU — OUTER FOLD 5/5


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor preprocessing: stem
F1-Macro train: 0.8137
Mejor F1 inner: 0.6725
F1-Macro outer: 0.6267
Gap train-outer: 0.1870

RESULTADOS NESTED CV 5×5


,variante,f1_train_mean,f1_nested_mean,f1_nested_std,f1_nested_min,f1_nested_max,gap_mean
0,mx,0.7864,0.5951,0.0238,0.5730,0.6289,0.1913
1,es,0.8604,0.7034,0.0228,0.6786,0.7298,0.1570
2,cu,0.8020,0.6504,0.0278,0.6267,0.6956,0.1516


In [7]:
# Detalle de los 5 Outer folds por variante

for variante in VARIANTES:
    print(f"\nDETALLE OUTER — {variante.upper()}")
    display(
        nested_detalle[variante][[
            "outer_fold",
            "best_preprocessing",
            "train_f1_macro",
            "inner_best_f1",
            "inner_best_std",
            "outer_f1_macro",
            "generalization_gap",
        ]].style.format({
            "train_f1_macro": "{:.4f}",
            "inner_best_f1": "{:.4f}",
            "inner_best_std": "{:.4f}",
            "outer_f1_macro": "{:.4f}",
            "generalization_gap": "{:.4f}",
        })
    )

    print("\nMejor combinación por Outer Fold:")
    for _, row in nested_detalle[variante].iterrows():
        print(
            f"  Fold {int(row['outer_fold'])}: "
            f"prep={row['best_preprocessing']} | "
            f"params={row['best_params']}"
        )

    print("\nFrecuencia de preprocessing ganador:")
    print(nested_detalle[variante]["best_preprocessing"].value_counts())



DETALLE OUTER — MX


,outer_fold,best_preprocessing,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap
0,1,stem,0.7623,0.6091,0.0169,0.6085,0.1539
1,2,stem,0.9186,0.6229,0.0348,0.5910,0.3275
2,3,lemma,0.6989,0.6101,0.0215,0.6289,0.0700
3,4,stem,0.8295,0.6384,0.0173,0.5739,0.2555
4,5,stem,0.7226,0.6210,0.0262,0.5730,0.1495



Mejor combinación por Outer Fold:
  Fold 1: prep=stem | params={'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.07169245254517853, 'min_child_weight': 6, 'subsample': 0.8659255029128251, 'colsample_bytree': 0.5580442236513791, 'gamma': 3.7879173417095022, 'reg_alpha': 0.6785973360088753, 'reg_lambda': 3.022174564171313, 'scale_pos_weight': 1.470975033292052}
  Fold 2: prep=stem | params={'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.12441442056169186, 'min_child_weight': 2, 'subsample': 0.6950955297929516, 'colsample_bytree': 0.6139346810647384, 'gamma': 3.189895424999805, 'reg_alpha': 0.0010516118578332012, 'reg_lambda': 0.36488992787943136, 'scale_pos_weight': 2.2096165087649946}
  Fold 3: prep=lemma | params={'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.020302777462869792, 'min_child_weight': 3, 'subsample': 0.8280447675456006, 'colsample_bytree': 0.8305073003365188, 'gamma': 7.428839738290027, 'reg_alpha': 0.002739930072169357, 'reg_lambda': 2.55342093

,outer_fold,best_preprocessing,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap
0,1,lemma,0.8795,0.7265,0.0129,0.6786,0.2009
1,2,lemma,0.8777,0.7041,0.0127,0.7000,0.1777
2,3,lemma,0.8555,0.6997,0.0067,0.7237,0.1318
3,4,stem,0.8305,0.7145,0.0100,0.6849,0.1456
4,5,lemma,0.8588,0.7014,0.0299,0.7298,0.1290



Mejor combinación por Outer Fold:
  Fold 1: prep=lemma | params={'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.020754507947728467, 'min_child_weight': 2, 'subsample': 0.7410718065544867, 'colsample_bytree': 0.8515728334188893, 'gamma': 4.717822961536064, 'reg_alpha': 0.05236859953315358, 'reg_lambda': 1.7704128635726228, 'scale_pos_weight': 1.4787016994094708}
  Fold 2: prep=lemma | params={'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.03408579849231539, 'min_child_weight': 4, 'subsample': 0.7516689542646373, 'colsample_bytree': 0.5732370205684211, 'gamma': 0.7091586979048929, 'reg_alpha': 0.001905248583376566, 'reg_lambda': 7.368693814243105, 'scale_pos_weight': 1.3046516291729233}
  Fold 3: prep=lemma | params={'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.13818533275798747, 'min_child_weight': 3, 'subsample': 0.7010846769965768, 'colsample_bytree': 0.7124762700247501, 'gamma': 4.816487551932321, 'reg_alpha': 0.020710491516222514, 'reg_lambda': 2.117615

,outer_fold,best_preprocessing,train_f1_macro,inner_best_f1,inner_best_std,outer_f1_macro,generalization_gap
0,1,lemma,0.7304,0.6527,0.0395,0.6308,0.0996
1,2,stem,0.8487,0.6603,0.0145,0.6956,0.1531
2,3,lemma,0.7748,0.6467,0.0342,0.6559,0.1189
3,4,stem,0.8422,0.6654,0.0146,0.6427,0.1996
4,5,stem,0.8137,0.6725,0.0202,0.6267,0.1870



Mejor combinación por Outer Fold:
  Fold 1: prep=lemma | params={'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.09443178938150859, 'min_child_weight': 3, 'subsample': 0.8692138929684563, 'colsample_bytree': 0.7546143737544789, 'gamma': 7.694003892709915, 'reg_alpha': 0.009303604611881816, 'reg_lambda': 14.937435643972782, 'scale_pos_weight': 1.9964379625056565}
  Fold 2: prep=stem | params={'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01218134534286882, 'min_child_weight': 2, 'subsample': 0.6984424282572805, 'colsample_bytree': 0.5561207876591674, 'gamma': 7.465145517879115, 'reg_alpha': 0.021565234387475433, 'reg_lambda': 0.1803633360847891, 'scale_pos_weight': 1.8960779084638661}
  Fold 3: prep=lemma | params={'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.03493473556146263, 'min_child_weight': 4, 'subsample': 0.7320793831619755, 'colsample_bytree': 0.5155449178927876, 'gamma': 7.123727766702909, 'reg_alpha': 0.003193027275121406, 'reg_lambda': 0.8802804

## 7. Optimización final sobre todo el train

Nested CV sirve para estimar la generalización del procedimiento completo.

Una vez cerrada esa evaluación, se ejecuta una nueva búsqueda sobre **todo el train oficial** para obtener la combinación definitiva:

- preprocessing;
- hiperparámetros XGBoost.

El test oficial todavía no participa.


In [8]:
def optimizar_final_variante(variante, n_trials=N_TRIALS_FINAL):
    X_por_prep, y = cargar_todos_preps(variante, split="train")

    cv_final = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    final_splits = list(cv_final.split(X_por_prep["normal"], y))

    def objective(trial):
        prep = trial.suggest_categorical(
            "preprocessing", list(PREPROCESAMIENTOS.keys())
        )
        params = sugerir_params_xgb(trial)
        pipeline = crear_pipeline(params, random_state=RANDOM_STATE)

        scores = cross_val_score(
            estimator=pipeline,
            X=X_por_prep[prep],
            y=y,
            cv=final_splits,
            scoring="f1_macro",
            n_jobs=1,
        )
        trial.set_user_attr("cv_f1_std", float(scores.std(ddof=1)))
        return float(scores.mean())

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )
    return study


studies_finales = {}

for variante in VARIANTES:
    print(f"\n{'='*70}")
    print(f"OPTIMIZACIÓN FINAL — {variante.upper()}")
    print(f"{'='*70}")

    study = optimizar_final_variante(
        variante, n_trials=N_TRIALS_FINAL
    )
    studies_finales[variante] = study

    print(f"Mejor F1-CV: {study.best_value:.4f}")
    print(f"Mejor preprocessing: {study.best_params['preprocessing']}")
    print("Mejores hiperparámetros XGBoost:")
    for k, v in study.best_params.items():
        if k != "preprocessing":
            print(f"  {k}: {repr(v)}")



OPTIMIZACIÓN FINAL — MX


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor F1-CV: 0.6244
Mejor preprocessing: normal
Mejores hiperparámetros XGBoost:
  n_estimators: 650
  max_depth: 3
  learning_rate: 0.02992226863094147
  min_child_weight: 3
  subsample: 0.7015929604579353
  colsample_bytree: 0.6700163019309586
  gamma: 3.206142668165488
  reg_alpha: 0.6992470454250899
  reg_lambda: 0.15834451867267532
  scale_pos_weight: 1.7909291838347168

OPTIMIZACIÓN FINAL — ES


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor F1-CV: 0.7049
Mejor preprocessing: normal
Mejores hiperparámetros XGBoost:
  n_estimators: 600
  max_depth: 3
  learning_rate: 0.013567662411096632
  min_child_weight: 3
  subsample: 0.7526965914574834
  colsample_bytree: 0.5509446456627474
  gamma: 4.67042234219341
  reg_alpha: 1.039096427652307
  reg_lambda: 0.2471044820643491
  scale_pos_weight: 1.4410971672069746

OPTIMIZACIÓN FINAL — CU


  0%|          | 0/50 [00:00<?, ?it/s]

Mejor F1-CV: 0.6650
Mejor preprocessing: stem
Mejores hiperparámetros XGBoost:
  n_estimators: 650
  max_depth: 4
  learning_rate: 0.021725423114650193
  min_child_weight: 3
  subsample: 0.7751612513093522
  colsample_bytree: 0.5344434552671052
  gamma: 7.277132342194694
  reg_alpha: 0.045093940109743186
  reg_lambda: 0.5701154663714562
  scale_pos_weight: 1.668412382733185


In [9]:
# Resumen de la selección final

filas = []
for variante, study in studies_finales.items():
    fila = {
        "variante": variante,
        "best_f1_cv_final": study.best_value,
        "cv_std_final": study.best_trial.user_attrs.get("cv_f1_std", np.nan),
    }
    fila.update(study.best_params)
    filas.append(fila)


df_params_finales = pd.DataFrame(filas)

display(
    df_params_finales.style.format({
        "best_f1_cv_final": "{:.4f}",
        "cv_std_final": "{:.4f}",
        "learning_rate": "{:.6f}",
        "subsample": "{:.4f}",
        "colsample_bytree": "{:.4f}",
        "gamma": "{:.4f}",
        "reg_alpha": "{:.6f}",
        "reg_lambda": "{:.6f}",
        "scale_pos_weight": "{:.4f}",
    })
)


,variante,best_f1_cv_final,cv_std_final,preprocessing,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,scale_pos_weight
0,mx,0.6244,0.0144,normal,650,3,0.029922,3,0.7016,0.6700,3.2061,0.699247,0.158345,1.7909
1,es,0.7049,0.0214,normal,600,3,0.013568,3,0.7527,0.5509,4.6704,1.039096,0.247104,1.4411
2,cu,0.6650,0.0183,stem,650,4,0.021725,3,0.7752,0.5344,7.2771,0.045094,0.570115,1.6684


## 8. Guardar resultados antes de tocar el test

Se guardan el resumen Nested CV, el detalle de cada Outer Fold, el preprocessing ganador, los mejores hiperparámetros y el historial completo de trials de Optuna.


In [10]:
os.makedirs(DATA_DIR, exist_ok=True)

df_nested_resumen.to_csv(
    f"{DATA_DIR}/xgboost_nestedcv_prep_inner_resumen.csv",
    index=False,
)

for variante in VARIANTES:
    nested_detalle[variante].drop(columns=["best_params"]).to_csv(
        f"{DATA_DIR}/xgboost_nestedcv_prep_inner_detalle_{variante}.csv",
        index=False,
    )

    outer_configs = {}
    for _, row in nested_detalle[variante].iterrows():
        outer_configs[str(int(row["outer_fold"]))] = {
            "preprocessing": row["best_preprocessing"],
            "params": row["best_params"],
        }

    with open(
        f"{DATA_DIR}/xgboost_nestedcv_prep_inner_best_{variante}.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(outer_configs, f, indent=4)

    for outer_fold, study in nested_studies[variante].items():
        study.trials_dataframe().to_csv(
            f"{DATA_DIR}/xgboost_optuna_prep_inner_{variante}_outer{outer_fold}.csv",
            index=False,
        )

best_final_json = {}
for variante, study in studies_finales.items():
    params = study.best_params.copy()
    prep = params.pop("preprocessing")
    best_final_json[variante] = {
        "preprocessing": prep,
        "params": params,
        "best_f1_cv": study.best_value,
    }

with open(
    f"{DATA_DIR}/xgboost_best_prep_params_final.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(best_final_json, f, indent=4)

for variante, study in studies_finales.items():
    study.trials_dataframe().to_csv(
        f"{DATA_DIR}/xgboost_optuna_final_prep_inner_{variante}.csv",
        index=False,
    )

print("Resultados y diagnósticos guardados.")


Resultados y diagnósticos guardados.


### 8.1 Qué comparar con la corrida anterior

Antes de ejecutar el test oficial, compara esta nueva corrida contra el baseline anterior usando exclusivamente Nested CV:

- `f1_nested_mean`: rendimiento externo medio;
- `f1_nested_std`: estabilidad entre folds;
- `gap_mean`: diferencia media Train–Outer;
- frecuencia de `best_preprocessing`: estabilidad de la decisión de preprocessing.

No elijas la mejor versión mirando el test oficial.

In [11]:
comparacion_actual = df_nested_resumen[[
    "variante",
    "f1_train_mean",
    "f1_nested_mean",
    "f1_nested_std",
    "f1_nested_min",
    "f1_nested_max",
    "gap_mean",
]].copy()

print("NUEVA CORRIDA — PREPROCESSING DENTRO DEL INNER CV")
display(
    comparacion_actual.style.format({
        "f1_train_mean": "{:.4f}",
        "f1_nested_mean": "{:.4f}",
        "f1_nested_std": "{:.4f}",
        "f1_nested_min": "{:.4f}",
        "f1_nested_max": "{:.4f}",
        "gap_mean": "{:.4f}",
    })
)


NUEVA CORRIDA — PREPROCESSING DENTRO DEL INNER CV


,variante,f1_train_mean,f1_nested_mean,f1_nested_std,f1_nested_min,f1_nested_max,gap_mean
0,mx,0.7864,0.5951,0.0238,0.5730,0.6289,0.1913
1,es,0.8604,0.7034,0.0228,0.6786,0.7298,0.1570
2,cu,0.8020,0.6504,0.0278,0.6267,0.6956,0.1516


## 9. Entrenamiento final y evaluación en test oficial

Ejecuta esta sección **solo cuando la Nested CV y la optimización final estén cerradas**.

Cada variante usa el preprocessing y los hiperparámetros seleccionados en la optimización final sobre todo el train. El test oficial se evalúa una sola vez.


In [12]:
def evaluar_test_final(variante):
    best_all = studies_finales[variante].best_params.copy()
    prep = best_all.pop("preprocessing")
    best_params = best_all

    train = cargar_split(variante, prep, split="train")
    test = cargar_split(variante, prep, split="test")

    X_train, y_train = preparar_xy(train)
    X_test, y_test = preparar_xy(test)

    pipeline = crear_pipeline(best_params, random_state=RANDOM_STATE)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    resultado = {
        "variante": variante,
        "preprocesamiento": prep,
        "f1_macro_test": f1_score(y_test, y_pred, average="macro"),
        "accuracy_test": accuracy_score(y_test, y_pred),
        "precision_macro_test": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "recall_macro_test": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
    }
    return resultado, pipeline, y_test, y_pred


In [13]:

resultados_test = []
modelos_finales = {}

for variante in VARIANTES:
    print(f"\n{'='*70}")
    print(f"TEST OFICIAL — {variante.upper()}")
    print(f"{'='*70}")

    resultado, pipeline, y_test, y_pred = evaluar_test_final(
        variante
    )

    resultados_test.append(resultado)
    modelos_finales[variante] = pipeline

    print(f"F1-Macro: {resultado['f1_macro_test']:.4f}")
    print(f"Accuracy: {resultado['accuracy_test']:.4f}")
    print(f"Precision Macro: {resultado['precision_macro_test']:.4f}")
    print(f"Recall Macro: {resultado['recall_macro_test']:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            digits=4,
            zero_division=0,
        )
    )

    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred))


df_resultados_test = pd.DataFrame(resultados_test)

print("\nRESULTADOS FINALES XGBOOST — TEST OFICIAL")
display(
    df_resultados_test.style.format({
        "f1_macro_test": "{:.4f}",
        "accuracy_test": "{:.4f}",
        "precision_macro_test": "{:.4f}",
        "recall_macro_test": "{:.4f}",
    })
)



TEST OFICIAL — MX
F1-Macro: 0.6230
Accuracy: 0.6450
Precision Macro: 0.6230
Recall Macro: 0.6357

Classification report:
              precision    recall  f1-score   support

           0     0.7733    0.6633    0.7141       401
           1     0.4727    0.6080    0.5319       199

    accuracy                         0.6450       600
   macro avg     0.6230    0.6357    0.6230       600
weighted avg     0.6736    0.6450    0.6537       600

Matriz de confusión:
[[266 135]
 [ 78 121]]

TEST OFICIAL — ES
F1-Macro: 0.6918
Accuracy: 0.7250
Precision Macro: 0.6911
Recall Macro: 0.6925

Classification report:
              precision    recall  f1-score   support

           0     0.7960    0.7900    0.7930       400
           1     0.5862    0.5950    0.5906       200

    accuracy                         0.7250       600
   macro avg     0.6911    0.6925    0.6918       600
weighted avg     0.7260    0.7250    0.7255       600

Matriz de confusión:
[[316  84]
 [ 81 119]]

TEST OFICIAL 

,variante,preprocesamiento,f1_macro_test,accuracy_test,precision_macro_test,recall_macro_test
0,mx,normal,0.6230,0.6450,0.6230,0.6357
1,es,normal,0.6918,0.7250,0.6911,0.6925
2,cu,stem,0.6551,0.7067,0.6653,0.6500


In [14]:
# Guardar resultados finales del test

df_resultados_test.to_csv(
    f"{DATA_DIR}/xgboost_test_final_prep_inner.csv",
    index=False,
)

print(f"Guardado en {DATA_DIR}/xgboost_test_final_prep_inner.csv")


Guardado en ../data/xgboost_test_final_prep_inner.csv
